# Lab: **Running & Quantizing `llama.cpp` Models on Android**

This hands-on lab walks you through the complete workflow of taking an open LLaMA‑style model, converting and quantizing it with `llama.cpp`, and finally running it natively on both your desktop **and** an Android phone (Pixel).  
You will:

1. **Clone & build** `llama.cpp`
2. **Download** a compact LLaMA model from HuggingFace
3. **Convert** it to GGUF
4. **Quantize** to multiple bit‑widths
5. **Run** locally on your computer
6. **Cross‑compile** `llama.cpp` for Android
7. **Deploy & run** the model on a real device

---

## ⚙️ Prerequisites

| Tool / SDK | Version (tested) | Notes |
|------------|-----------------|-------|
| **Python** | ≥ 3.9           | Needed for model download & conversion |
| **CMake**  | ≥ 3.16          | Build system generator |
| **GCC/Clang** | Recent       | For native compilation |
| **Git**    | Any            | Clone repositories |
| **ADB** + **Android NDK** | r29 or newer | Cross‑compiling & pushing binaries |

> **Tip:** All commands are shown for macOS/Linux. On Windows, use **WSL 2** or adapt paths accordingly.

---

## 1 Clone `llama.cpp`

```bash
git clone https://github.com/ggml-org/llama.cpp.git
```

---

## 2 Create a Workspace for Models

```bash
mkdir models
cd models
```

---

## 3 Download a HuggingFace Model

Create `download_model.py`:

```python
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="HuggingFaceTB/SmolLM2-360M-Instruct",
    local_dir="SmolLM2-360M-Instruct",
    local_dir_use_symlinks=False
)
```

Run:

```bash
python download_model.py
```

The **360 M parameter** size keeps build times short while still demonstrating all steps.

---

## 4 Convert to GGUF

```bash
cd llama.cpp
python convert_hf_to_gguf.py ../models/SmolLM2-360M-Instruct --outfile ../models/SmolLM2-360M-Instruct-gguf
```

> **Why GGUF?**  
> GGUF is the native on‑disk format for `llama.cpp`, enabling efficient mmap‑style loading and platform‑agnostic quantization.

---

## 5 Build `llama.cpp` (Native)

```bash
mkdir build-native
cd build-native
cmake .. -DCMAKE_BUILD_TYPE=Release -DLLAMA_BUILD_TESTS=OFF -DLLAMA_BUILD_EXAMPLES=OFF -DLLAMA_BUILD_TOOLS=ON
make -j4
```

The `llama-run` binary (plus `llama-quantize`, etc.) will appear in `build-native/bin`.

---

## 6 Quantize the Model

```bash
cd ../
./build-native/bin/llama-quantize ../models/SmolLM2-360M-Instruct-gguf ../models/SmolLM2-360M-Instruct-gguf-Q8_0 Q8_0
./build-native/bin/llama-quantize ../models/SmolLM2-360M-Instruct-gguf ../models/SmolLM2-360M-Instruct-gguf-Q4_K_M Q4_K_M
./build-native/bin/llama-quantize ../models/SmolLM2-360M-Instruct-gguf ../models/SmolLM2-360M-Instruct-gguf-TQ2_0 TQ2_0
```

* **Q8_0** ≈ 8‑bit (baseline quality)  
* **Q4_K_M** ≈ 4‑bit (fast, good perplexity)  
* **TQ2_0** ≈ 2‑bit “true” quantization (tiny footprint)

---

## 7 Run a Quantized Model Locally

```bash
./build-native/bin/llama-run ../models/SmolLM2-360M-Instruct-gguf-Q8_0 "Hello, could you describe to me how quantization works?"
```

Expect a short inference delay the first time while the model is paged in.

---

## 8 Set Up for Android Cross‑Compilation

1. **Install** ADB + NDK via Android Studio (SDK Manager ▶ NDK).  
2. **Locate** your NDK root, e.g.:

   ```text
   ~/Library/Android/sdk/ndk/29.0.13599879/
   ```

3. **Verify** the CMake toolchain file exists:

   ```text
   ~/Library/Android/sdk/ndk/29.0.13599879/build/cmake/android.toolchain.cmake
   ```

---

## 9 Build `llama.cpp` for Android (arm64‑v8a)

```bash
mkdir build-android
cd build-android

cmake ..   -DCMAKE_TOOLCHAIN_FILE=~/Library/Android/sdk/ndk/29.0.13599879/build/cmake/android.toolchain.cmake   -DANDROID_ABI=arm64-v8a   -DANDROID_PLATFORM=android-23   -DCMAKE_BUILD_TYPE=Release   -DLLAMA_NATIVE=OFF   -DLLAMA_BUILD_TESTS=OFF   -DLLAMA_BUILD_EXAMPLES=OFF   -DLLAMA_CURL=OFF   -DGGML_OPENMP=OFF

make -j4
```

The resulting binaries land in `build-android/bin`.

---

## 10 Connect Device & Verify ADB

```bash
adb devices
```

If your Pixel is listed, you’re good to go.

---

## 11 Script to Push Binaries & Model

Create `push_model.sh`:

```bash
#!/bin/bash

# Paths — edit if needed
LLAMA_CPP_DIR=/Users/olivergrainge/github/test/llama.cpp
TARGET_DIR=/data/local/tmp/llama
BUILD_DIR=$LLAMA_CPP_DIR/build-android/bin

# Read model path from first command line argument
if [ -z "$1" ]; then
    echo "Usage: $0 path/to/model.gguf"
    exit 1
fi

MODEL_PATH="$1"

echo "Model to push: $MODEL_PATH"

# Determine correct llama binary name
LLAMA_BIN=llama-run
if [ ! -f $BUILD_DIR/llama-run ] && [ -f $BUILD_DIR/main ]; then
    LLAMA_BIN=main
fi

echo "Pushing $LLAMA_BIN..."
adb shell mkdir -p $TARGET_DIR
adb push $BUILD_DIR/$LLAMA_BIN $TARGET_DIR/llama-run

echo "Pushing libraries (optional)..."
for lib in libllama.so libggml.so libggml-cpu.so libggml-base.so; do
    if [ -f $BUILD_DIR/$lib ]; then
        adb push $BUILD_DIR/$lib $TARGET_DIR/$lib
    fi
done

echo "Pushing model..."
adb push $MODEL_PATH $TARGET_DIR/model.gguf

echo "Done."
```

Make it executable & run:

```bash
chmod +x push_model.sh
./push_model.sh models/SmolLM2-360M-Instruct-gguf-Q8_0
```

---

## 12 Run On‑Device 🎉

```bash
adb shell
cd /data/local/tmp/llama
chmod +x llama-run
LD_LIBRARY_PATH=. ./llama-run model.gguf "Hello, could you describe to me how quantization works?"
```

You should see the model respond directly on the handset, with no network required.

---

## ✅ Recap

- **Built** `llama.cpp` for both x86‑64 and arm64‑v8a  
- **Converted & quantized** a 360 M parameter model to GGUF (Q8/Q4/TQ2)  
- **Executed** the model locally on desktop  
- **Cross‑compiled, deployed, and executed** on an Android Pixel phone  

You now have a self‑contained workflow for experimenting with small LLMs on‑device. Happy hacking! 🚀